# 🚗 Car Price Prediction — Lasso Regression
Predicting the resale price of used cars using machine learning.

**Pipeline:**
1. Import dependencies
2. Load & explore data
3. EDA & visualizations
4. Feature engineering
5. Train / evaluate model
6. Save model

## 1. Import Dependencies

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import Lasso, LinearRegression
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_theme(style='darkgrid', palette='muted')
print('All dependencies loaded ✅')

## 2. Load & Explore Data

In [ ]:
df = pd.read_csv('car.csv')
print(f'Shape: {df.shape}')
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
print('Missing values:')
print(df.isnull().sum())

In [ ]:
print('Fuel Type:\n', df['Fuel_Type'].value_counts())
print('\nSeller Type:\n', df['Seller_Type'].value_counts())
print('\nTransmission:\n', df['Transmission'].value_counts())

## 3. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

df['Fuel_Type'].value_counts().plot(kind='bar', ax=axes[0], color=['#667eea','#764ba2','#f093fb'])
axes[0].set_title('Fuel Type Distribution')
axes[0].set_xlabel('')

df['Seller_Type'].value_counts().plot(kind='bar', ax=axes[1], color=['#667eea','#764ba2'])
axes[1].set_title('Seller Type Distribution')
axes[1].set_xlabel('')

df['Transmission'].value_counts().plot(kind='bar', ax=axes[2], color=['#667eea','#764ba2'])
axes[2].set_title('Transmission Distribution')
axes[2].set_xlabel('')

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['Selling_Price'], bins=30, color='#667eea', edgecolor='white')
axes[0].set_title('Selling Price Distribution')
axes[0].set_xlabel('Price (Lakhs ₹)')

axes[1].scatter(df['Present_Price'], df['Selling_Price'], alpha=0.5, color='#764ba2')
axes[1].set_title('Present Price vs Selling Price')
axes[1].set_xlabel('Present Price (Lakhs ₹)')
axes[1].set_ylabel('Selling Price (Lakhs ₹)')

plt.tight_layout()
plt.show()

In [ ]:
# Average selling price by year
price_by_year = df.groupby('Year')['Selling_Price'].mean()
plt.figure(figsize=(12, 4))
price_by_year.plot(marker='o', color='#667eea', linewidth=2)
plt.title('Average Selling Price by Year')
plt.xlabel('Year')
plt.ylabel('Avg Selling Price (Lakhs ₹)')
plt.tight_layout()
plt.show()

## 4. Feature Engineering

In [ ]:
# Encode categorical columns
df_encoded = df.copy()
df_encoded.replace({'Fuel_Type':    {'Petrol': 0, 'Diesel': 1, 'CNG': 2}}, inplace=True)
df_encoded.replace({'Seller_Type':  {'Dealer': 0, 'Individual': 1}},        inplace=True)
df_encoded.replace({'Transmission': {'Manual': 0, 'Automatic': 1}},         inplace=True)

# Engineered feature: car age is more intuitive than raw year
df_encoded['Car_Age'] = 2024 - df_encoded['Year']

df_encoded.head()

In [ ]:
# Correlation heatmap (numeric columns only)
numeric_df = df_encoded.drop(columns=['Car_Name'])
plt.figure(figsize=(10, 7))
sns.heatmap(numeric_df.corr(), annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5)
plt.title('Feature Correlation Heatmap')
plt.tight_layout()
plt.show()

In [ ]:
# Features & target
X = df_encoded[['Year', 'Car_Age', 'Present_Price', 'Kms_Driven', 'Fuel_Type', 'Seller_Type', 'Transmission', 'Owner']]
y = df_encoded['Selling_Price']

print('Features:', list(X.columns))
print('X shape:', X.shape)
print('y shape:', y.shape)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Train: {X_train.shape} | Test: {X_test.shape}')

## 5. Train & Evaluate Model

We compare Lasso vs Gradient Boosting. The original Lasso (alpha=1.0) zeroed out 4/7 features due to over-regularization — Gradient Boosting is far more suitable here.

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

# Baseline: Lasso (original)
lasso_orig = Lasso(alpha=1.0)
lasso_orig.fit(X_train, y_train)
print('Lasso coefficients:', dict(zip(X.columns, lasso_orig.coef_.round(4))))
print('(Notice: 4 features zeroed out by over-regularization)')

In [ ]:
# Improved model: Gradient Boosting
model = GradientBoostingRegressor(n_estimators=200, learning_rate=0.05, max_depth=4, random_state=42)
model.fit(X_train, y_train)
print('Gradient Boosting trained ✅')

In [ ]:
def evaluate(m, X_tr, X_te, y_tr, y_te, name):
    for split, Xd, yd in [('Train', X_tr, y_tr), ('Test', X_te, y_te)]:
        pred = m.predict(Xd)
        print(f'[{name} | {split}]  R²={r2_score(yd, pred):.4f}  '
              f'MAE={mean_absolute_error(yd, pred):.3f}  '
              f'RMSE={np.sqrt(mean_squared_error(yd, pred)):.3f}')

evaluate(lasso_orig, X_train, X_test, y_train, y_test, 'Lasso(α=1.0)     ')
evaluate(model,      X_train, X_test, y_train, y_test, 'GradientBoosting ')

In [ ]:
# Actual vs Predicted
y_pred_train = model.predict(X_train)
y_pred_test  = model.predict(X_test)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (y_true, y_pred, title) in zip(axes, [
    (y_train, y_pred_train, 'Train Set'),
    (y_test,  y_pred_test,  'Test Set')
]):
    ax.scatter(y_true, y_pred, alpha=0.6, color='#667eea', edgecolors='white', linewidths=0.3)
    lims = [min(y_true.min(), y_pred.min()), max(y_true.max(), y_pred.max())]
    ax.plot(lims, lims, 'r--', linewidth=1.5, label='Perfect Prediction')
    ax.set_title(f'Actual vs Predicted — {title}')
    ax.set_xlabel('Actual Price (Lakhs ₹)')
    ax.set_ylabel('Predicted Price (Lakhs ₹)')
    ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Feature importance
imp_df = pd.DataFrame({'Feature': X.columns, 'Importance': model.feature_importances_})
imp_df = imp_df.sort_values('Importance')

plt.figure(figsize=(8, 5))
plt.barh(imp_df['Feature'], imp_df['Importance'], color='#667eea')
plt.title('Gradient Boosting — Feature Importance')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

## 6. Save Model

In [ ]:
with open('car_price_model.pkl', 'wb') as f:
    pickle.dump(model, f)
print('Model saved as car_price_model.pkl ✅')

In [ ]:
# Sanity check — reload and predict
with open('car_price_model.pkl', 'rb') as f:
    loaded = pickle.load(f)

# 2015 car, ₹5L showroom, 50k km, Petrol, Dealer, Manual, 0 owners, age=9
sample = pd.DataFrame([[2015, 9, 5.0, 50000, 0, 0, 0, 0]],
    columns=['Year','Car_Age','Present_Price','Kms_Driven','Fuel_Type','Seller_Type','Transmission','Owner'])
print(f'Sample prediction: ₹ {loaded.predict(sample)[0]:.2f} Lakhs')